# EfficientAD Anomaly Detection: Master Reproduction Notebook

Reproduces the EfficientAD pipeline for the ADL 2025-2026 challenge.

Pipeline (single script `scripts/train_evaluate_efficient_ad_v2.py`):
1. **Training** — pre-distilled teacher PDN + trainable student + autoencoder. Top-10% hardest-pixel student loss, AE MSE loss, penalty loss on STL10 (so the student doesn't mimic the teacher OOD), plus an auxiliary margin loss on the few labeled anomaly masks.
2. **Quantile calibration** — computes per-stream `q_a/q_b` on the normal images so the ST and AE maps are normalized to comparable scales before summing.
3. **Local evaluation + PDF** — anomaly maps on the labeled training anomalies, for manual inspection (NOTE: training-set overlap, optimistic).
4. **Submission** — per-class percentile-clipped + multi-view sample gate → q8rle CSV/ZIP.

**Teacher checkpoint:** Anomalib's pre-distilled teacher (`pretrained_teacher_small.pth`, out_channels=384) is required. If it's not present at `./checkpoints/efficient_ad/teacher_small.pth`, the model auto-downloads it from the Anomalib GitHub release on first instantiation — no manual setup needed.

**Note:** Designed to run on Google Colab or locally with the correct environment.

## 1. Setup and Environment

In [ ]:
!nvidia-smi

In [ ]:
# If running on Colab, uncomment and run this:
# !pip install timm opencv-python pandas tqdm matplotlib scipy torchvision

In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

Current working directory: /home/max/Documents/ghi8/adl/anomaly-detection


### Teacher checkpoint (auto-download)

The first time `EfficientAD(...)` is instantiated, if the teacher checkpoint isn't present it is downloaded from the Anomalib GitHub release:

- URL: `https://github.com/open-edge-platform/anomalib/releases/download/efficientad_pretrained_weights/efficientad_pretrained_weights.zip`
- File extracted: `efficientad_pretrained_weights/pretrained_teacher_small.pth`
- Saved to: `./checkpoints/efficient_ad/teacher_small.pth`

You can pre-trigger the download (or just verify presence) by running the next cell. If you already have the file at a different path, pass `--teacher_ckpt <path>` to the run cell below.

In [2]:
from pathlib import Path
ckpt = Path("checkpoints/efficient_ad/teacher_small.pth")
if ckpt.exists():
    print(f"Teacher checkpoint present: {ckpt} ({ckpt.stat().st_size / 1e6:.2f} MB)")
else:
    print(f"Teacher checkpoint missing at {ckpt} — will be auto-downloaded on first EfficientAD(...) call.")
    print("To pre-download now, uncomment the next two lines:")
    print("# from adl_lib.efficient_ad import EfficientAD")
    print("# _ = EfficientAD()  # triggers download, then discard")

Teacher checkpoint present: checkpoints/efficient_ad/teacher_small.pth (10.78 MB)


## 2. Execution Pipeline

The v2 script runs training (with STL10 penalty stream — STL10 auto-downloads to `data/stl10/` on first run, ~13MB), quantile calibration, local evaluation + PDF, and submission encoding in one pass.

Tweakable flags:
- `--epochs` (default 15)
- `--lr` (default 1e-4)
- `--teacher_ckpt` (default `./checkpoints/efficient_ad/teacher_small.pth`)
- `--student_topk` (default 0.1) — fraction of hardest pixels used for the student loss
- `--w_aux` (default 0.05) — weight on the auxiliary supervised mask loss; set to 0 to disable
- `--view_gate_alpha` (default 0.5) — sample-level multi-view multiplicative gate strength; set to 0 to disable
- `--classes class_01 class_02 ...` (default: all classes found under `PATH`)

Outputs:
- `artifacts/efficient_ad_v2/metrics.csv`
- `artifacts/efficient_ad_v2/predictions.pdf`
- `submission/submission_efficient_ad_v2.csv`
- `submission/submission_efficient_ad_v2.zip`

In [3]:
# Train + quantile calibration + local eval + PDF + submission (one pass)
%run scripts/train_evaluate_efficient_ad_v2.py

Processing 8 classes: ['class_01', 'class_02', 'class_03', 'class_04', 'class_05', 'class_06', 'class_07', 'class_08']

>>> Class: class_01
Training on 2600 good + 25 labeled anomaly samples...


Calibrating per-stream map quantiles on normal images...
Evaluating on 25 labeled anomaly samples (NOTE: training-set overlap)...


[train-set eval] Pixel AP: 0.3428 | Pixel AUROC: 0.9381
Predicting on 465 test samples...



>>> Class: class_02
Training on 2135 good + 30 labeled anomaly samples...


Calibrating per-stream map quantiles on normal images...
Evaluating on 30 labeled anomaly samples (NOTE: training-set overlap)...


...

[train-set eval] Pixel AP: 0.1477 | Pixel AUROC: 0.6272
Predicting on 915 test samples...



>>> Class: class_08
Training on 2045 good + 35 labeled anomaly samples...


Calibrating per-stream map quantiles on normal images...
Evaluating on 35 labeled anomaly samples (NOTE: training-set overlap)...


[train-set eval] Pixel AP: 0.7726 | Pixel AUROC: 0.9167
Predicting on 960 test samples...



Saved metrics to artifacts/efficient_ad_v2/metrics.csv
Mean Pixel AP (train-set eval, optimistic): 0.3470
Mean Pixel AUROC (train-set eval, optimistic): 0.8483
Saved PDF to artifacts/efficient_ad_v2/predictions.pdf
Saved submission to submission/submission_efficient_ad_v2.zip


## 3. Kaggle Submission

Uncomment and run the following cell to submit the results directly to Kaggle.

In [4]:
# !kaggle competitions submit -c adl-2025-2026-anomaly-detection -f submission/submission_efficient_ad_v2.zip -m "EfficientAD v2 (pre-distilled teacher + penalty + ST/AE quantile norm + view gate)"

100%|████████████████████████████████████████| 156M/156M [00:05<00:00, 30.1MB/s]
Successfully submitted to ADL [2025-2026]: Anomaly Detection